In [2]:
#Kalibratie

# --- Import ---
import os
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy import signal
from datetime import datetime, timedelta

# Bestandspaden
excel_path = r"C:\Users\Gebruiker\Documents\DATA\Vergelijking\SLB01008.xlsx"
wav_path   = r"C:\Users\Gebruiker\Documents\DATA\Vergelijking\20250901_135042.WAV"

# Tijdspanne waarin beide apparaten samenmetingen deden
start_time = datetime.strptime("2025-09-01 13:54:00", "%Y-%m-%d %H:%M:%S")
end_time   = datetime.strptime("2025-09-01 13:56:00", "%Y-%m-%d %H:%M:%S")

# --- Controleer bestanden ---
if not os.path.exists(excel_path):
    raise FileNotFoundError(f"Bestand niet gevonden: {excel_path}")
if not os.path.exists(wav_path):
    raise FileNotFoundError(f"Bestand niet gevonden: {wav_path}")

# --- Lees Excel ---
ext = os.path.splitext(excel_path)[1].lower()
if ext == ".xlsx":
    df = pd.read_excel(excel_path)
elif ext == ".csv":
    df = pd.read_csv(excel_path, sep=None, engine="python")
else:
    raise ValueError("Bestand moet .xlsx of .csv zijn")

# --- Controleer verplichte kolommen ---
for col in ["Date", "Time"]:
    if col not in df.columns:
        raise ValueError(f"Kolom '{col}' ontbreekt in het bestand")

# --- Converteer Date en Time ---
def excel_date_to_datetime(excel_serial):
    if pd.isna(excel_serial):
        return None
    return datetime(1899, 12, 30) + timedelta(days=float(excel_serial))

if np.issubdtype(df["Date"].dtype, np.number):
    df["Date"] = df["Date"].apply(excel_date_to_datetime)
else:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

def parse_excel_time(val):
    if pd.isna(val):
        return timedelta(0)
    if isinstance(val, (int, float)):
        return timedelta(days=float(val))
    if isinstance(val, str):
        try:
            t = datetime.strptime(val.strip(), "%H:%M:%S").time()
            return timedelta(hours=t.hour, minutes=t.minute, seconds=t.second)
        except Exception:
            return timedelta(0)
    if hasattr(val, "hour"):
        return timedelta(hours=val.hour, minutes=val.minute, seconds=val.second)
    return timedelta(0)

df["datetime"] = df["Date"] + df["Time"].apply(parse_excel_time)

# --- Converteer Value naar numeriek ---
if "Value" not in df.columns:
    raise ValueError("Kolom 'Value' ontbreekt in het Excel-bestand.")

df["Value"] = (
    df["Value"]
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False)
)
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

if df["Value"].dropna().empty:
    raise ValueError("Geen geldige numerieke waarden gevonden in kolom 'Value'.")

# --- Filter data in opgegeven tijdsinterval ---
df_period = df[(df["datetime"] >= start_time) & (df["datetime"] <= end_time)]
if df_period.empty:
    raise ValueError("Geen meetdata binnen dit tijdsvenster")

# --- Gemiddelde waarde van de dBA-meter ---
db_meter_avg = df_period["Value"].mean()
print(f"Gemiddelde dBA-waarde geluidsmeter ({start_time} – {end_time}): {db_meter_avg:.2f} dBA")

# --- Lees WAV ---
sample_rate, audio_data = wavfile.read(wav_path)
audio_data = audio_data.astype(float)
audio_data /= np.max(np.abs(audio_data))  # normaliseer naar ±1

# Starttijd WAV uit bestandsnaam
wav_name = os.path.basename(wav_path)
wav_date_str, wav_time_str = wav_name.split(".")[0].split("_")
wav_start = datetime.strptime(wav_date_str + "_" + wav_time_str, "%Y%m%d_%H%M%S")

# Bereken indices voor het tijdsvenster
sample_start = int((start_time - wav_start).total_seconds() * sample_rate)
sample_end   = int((end_time   - wav_start).total_seconds() * sample_rate)
sample_start = max(sample_start, 0)
sample_end   = min(sample_end, len(audio_data))

if sample_start >= sample_end:
    raise ValueError("Tijdsvenster ligt buiten bereik van WAV-bestand")

audio_segment = audio_data[sample_start:sample_end]

# --- A-weging toepassen ---
def a_weighting_filter(x, fs):
    """Past een A-weging toe (IEC 61672) aan een audiosignaal."""
    # analoge A-wegingspolen en nulpunten
    f1, f2, f3, f4 = 20.598997, 107.65265, 737.86223, 12194.217
    A1000 = 1.9997

    NUMs = [(2 * np.pi * f4)**2 * (10**(A1000/20)), 0, 0, 0, 0]
    DENs = np.polymul([1, 4*np.pi*f4, (2*np.pi*f4)**2],
                      [1, 4*np.pi*f1, (2*np.pi*f1)**2])
    DENs = np.polymul(np.polymul(DENs,
                      [1, 2*np.pi*f3]),
                      [1, 2*np.pi*f2])

    # bilinear transformatie naar digitaal filter
    b, a = signal.bilinear(NUMs, DENs, fs)
    return signal.lfilter(b, a, x)

audio_a = a_weighting_filter(audio_segment, sample_rate)

# --- RMS en kalibratie ---
rms_a = np.sqrt(np.mean(audio_a ** 2))
db_wav_a = 20 * np.log10(rms_a)
calibration_offset = db_meter_avg - db_wav_a

print(f"Gemiddelde WAV-waarde (A-gewogen, ongekalibreerd): {db_wav_a:.2f} dBA")
print(f"Kalibratie-offset: {calibration_offset:.2f} dB")

results = {
    "periode_start": start_time,
    "periode_einde": end_time,
    "dBA_geluidsmeter": db_meter_avg,
    "dBA_wav": db_wav_a,
    "kalibratie_offset": calibration_offset,
}

print("\nSamenvatting:")
for k, v in results.items():
    print(f"{k:20s} : {v}")


Gemiddelde dBA-waarde geluidsmeter (2025-09-01 13:54:00 – 2025-09-01 13:56:00): 56.57 dBA
Gemiddelde WAV-waarde (A-gewogen, ongekalibreerd): -30.76 dBA
Kalibratie-offset: 87.33 dB

Samenvatting:
periode_start        : 2025-09-01 13:54:00
periode_einde        : 2025-09-01 13:56:00
dBA_geluidsmeter     : 56.56694214876033
dBA_wav              : -30.758943965269445
kalibratie_offset    : 87.32588611402977


C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_20500\3598466090.py:92: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sample_rate, audio_data = wavfile.read(wav_path)


In [10]:
# AudioMoth WAV → dBA-analyse per seconde (batch, met submappen)
#---------------------------------------------------
import numpy as np
import pandas as pd
import os
from scipy.io import wavfile
from scipy import signal
from datetime import datetime, timedelta

# === Configuratie van AudioMoth ===
SAMPLERATE_EXPECTED = 48000
GAIN = "Low"
GAIN_FACTORS = {
    "Low": 1.0,
    "Medium": 3.16,
    "High": 10.0,
}

# === Parameters ===
block_duration = 1.0  # analyse per seconde
CALIBRATION_OFFSET = 87.3259  # dB offset van dBFS → dBA (gekalibreerd)

# === A-wegingfilter ===
def a_weighting_filter(x, fs):
    """Past A-weging toe volgens IEC 61672."""
    f1, f2, f3, f4 = 20.598997, 107.65265, 737.86223, 12194.217
    A1000 = 1.9997

    NUMs = [(2 * np.pi * f4)**2 * (10**(A1000/20)), 0, 0, 0, 0]
    DENs = np.polymul([1, 4*np.pi*f4, (2*np.pi*f4)**2],
                      [1, 4*np.pi*f1, (2*np.pi*f1)**2])
    DENs = np.polymul(np.polymul(DENs,
                      [1, 2*np.pi*f3]),
                      [1, 2*np.pi*f2])
    b, a = signal.bilinear(NUMs, DENs, fs)
    return signal.lfilter(b, a, x)

# === Map met WAV-bestanden (inclusief submappen) ===
input_folder = r"C:\Users\Gebruiker\Documents\DATA\Audiomoth_3\ALL"
output_folder = r"C:\Users\Gebruiker\Documents\DATA\Audiomoth_3\output_dBA"
os.makedirs(output_folder, exist_ok=True)

# === Recursief door alle bestanden lopen ===
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.lower().endswith(".wav"):
            wav_file = os.path.join(root, file)
            print(f"🔹 Verwerken van: {wav_file}")

            # === WAV inlezen ===
            samplerate, data = wavfile.read(wav_file)
            if samplerate != SAMPLERATE_EXPECTED:
                print(f"⚠️ Waarschuwing: sample rate = {samplerate} Hz (verwacht {SAMPLERATE_EXPECTED} Hz)")

            # Gebruik enkel het eerste kanaal indien stereo
            if data.ndim > 1:
                data = data[:, 0]

            # Normaliseer naar ±1 (float)
            data = data.astype(float)
            data /= np.max(np.abs(data))

            # === A-weging toepassen ===
            data_a = a_weighting_filter(data, samplerate)

            # === Berekeningen ===
            block_size = int(samplerate * block_duration)
            n_blocks = len(data_a) // block_size

            rms_values = []
            for i in range(n_blocks):
                block = data_a[i * block_size:(i + 1) * block_size]
                rms = np.sqrt(np.mean(block**2))
                rms_values.append(rms)

            rms_values = np.array(rms_values)

            # === Stap 1: Bereken L_dBFS (A-gewogen) ===
            L_dBFS = 20 * np.log10(np.maximum(rms_values, 1e-10))

            # === Stap 2: Corrigeer voor microfoongain ===
            gain_factor = GAIN_FACTORS.get(GAIN, 1.0)
            L_dBFS_corrected = L_dBFS + 20 * np.log10(gain_factor)

            # === Stap 3: Converteer naar dBA via kalibratie-offset ===
            L_dBA = L_dBFS_corrected + CALIBRATION_OFFSET

            # === Afronden ===
            L_dBA = np.round(L_dBA, 4)
            L_dBFS_corrected = np.round(L_dBFS_corrected, 4)
            rms_values = np.round(rms_values, 6)

            # === Datum en starttijd uit bestandsnaam halen ===
            basename = os.path.splitext(file)[0]
            try:
                date_str, time_str = basename.split("_")
                start_datetime = datetime.strptime(date_str + time_str, "%Y%m%d%H%M%S")
            except ValueError:
                start_datetime = None

            # === Tijd per blok berekenen ===
            times = np.arange(n_blocks) * block_duration
            if start_datetime is not None:
                absolute_times = [start_datetime + timedelta(seconds=float(t)) for t in times]
            else:
                absolute_times = times

            # === DataFrame aanmaken ===
            df = pd.DataFrame({
                "absolute tijd": absolute_times,
                "tijd (s)": times,
                "RMS (A)": rms_values,
                "niveau (dBFS, A-gewogen)": L_dBFS_corrected,
                "niveau (dBA)": L_dBA,
                "samplerate (Hz)": samplerate,
                "gain": GAIN
            })

            # === Output-bestandsnaam ===
            output_csv_name = f"AM1_{basename}_A.csv"
            output_csv = os.path.join(output_folder, output_csv_name)
            df.to_csv(output_csv, index=False)
            print(f"✅ Bestand opgeslagen: {output_csv}\n")


🔹 Verwerken van: C:\Users\Gebruiker\Documents\DATA\Audiomoth_3\ALL\20250829\20250829_162055.WAV


C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_20500\3399158378.py:51: WavFileWarning: Chunk (non-data) not understood, skipping it.
  samplerate, data = wavfile.read(wav_file)


✅ Bestand opgeslagen: C:\Users\Gebruiker\Documents\DATA\Audiomoth_3\output_dBA\AM1_20250829_162055_A.csv

🔹 Verwerken van: C:\Users\Gebruiker\Documents\DATA\Audiomoth_3\ALL\20250829\20250829_163000.WAV
✅ Bestand opgeslagen: C:\Users\Gebruiker\Documents\DATA\Audiomoth_3\output_dBA\AM1_20250829_163000_A.csv

🔹 Verwerken van: C:\Users\Gebruiker\Documents\DATA\Audiomoth_3\ALL\20250829\20250829_164000.WAV
✅ Bestand opgeslagen: C:\Users\Gebruiker\Documents\DATA\Audiomoth_3\output_dBA\AM1_20250829_164000_A.csv

🔹 Verwerken van: C:\Users\Gebruiker\Documents\DATA\Audiomoth_3\ALL\20250829\20250829_165000.WAV
✅ Bestand opgeslagen: C:\Users\Gebruiker\Documents\DATA\Audiomoth_3\output_dBA\AM1_20250829_165000_A.csv

🔹 Verwerken van: C:\Users\Gebruiker\Documents\DATA\Audiomoth_3\ALL\20250829\20250829_170000.WAV
✅ Bestand opgeslagen: C:\Users\Gebruiker\Documents\DATA\Audiomoth_3\output_dBA\AM1_20250829_170000_A.csv

🔹 Verwerken van: C:\Users\Gebruiker\Documents\DATA\Audiomoth_3\ALL\20250829\20250829_1